# Bab 11: Regresi Logistik

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab11/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab11")
print("Folder kerja:", os.getcwd())

## Listing 11.1: Penurunan gradien untuk regresi logistik

`kode/bab11/gd_mini.py`

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

x = np.array([0.2, 0.5, 0.8, 1.0, 1.5, 2.5, 3.0, 4.5])
y = np.array([0, 0, 0, 1, 0, 1, 1, 1])
X = np.c_[np.ones_like(x), x]


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


b = np.zeros(2)
eta = 0.5
for it in range(1, 2001):
    p = sigmoid(X @ b)
    grad = X.T @ (p - y) / len(y)
    if it == 1:
        print("iterasi 1: p =", p.round(2), " gradien =", grad.round(4))
    b = b - eta * grad
    if it in (1, 2, 10, 100, 2000):
        ll = np.sum(y * np.log(sigmoid(X @ b))
                    + (1 - y) * np.log(1 - sigmoid(X @ b)))
        print(f"iterasi {it:4d}: b0 = {b[0]:+.4f}, b1 = {b[1]:+.4f},"
              f" log-lik = {ll:.4f}")
sk = LogisticRegression(penalty=None).fit(x[:, None], y)
print(f"scikit-learn: b0 = {sk.intercept_[0]:+.4f}, b1 = {sk.coef_[0, 0]:+.4f}")

## Listing 11.2: Koefisien dan rasio odds model churn

`kode/bab11/koefisien.py`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan.csv")
df["recency_10h"] = df["recency_hari"] / 10
df["tren"] = df["frekuensi_90h"] / (df["frekuensi_90h_lalu"] + 1)
df["promo_10pct"] = df["porsi_promo"] * 10
m = smf.logit("churn ~ recency_10h + tren + keluhan_90h + ragam_menu"
              " + promo_10pct + pakai_ulasan"
              " + C(tingkat, Treatment('perunggu'))", df).fit(disp=0)
tab = pd.DataFrame({"koef": m.params, "SE": m.bse, "p": m.pvalues,
                    "OR": np.exp(m.params),
                    "OR_bawah": np.exp(m.conf_int()[0]),
                    "OR_atas": np.exp(m.conf_int()[1])})
tab.index = [i.replace("C(tingkat, Treatment('perunggu'))[T.", "tingkat ")
             .replace("]", "") for i in tab.index]
print(tab.round(3).to_string())
print(f"pseudo-R2 McFadden = {m.prsquared:.3f}")

## Listing 11.3: Pilihan ambang dan efek pembobotan kelas

`kode/bab11/ambang.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab11')
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, "kode/bab07")
from siapkan import muat, pipa  # noqa: E402

X, y = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
biasa = pipa(LogisticRegression(max_iter=2000)).fit(Xl, yl)
seimbang = pipa(LogisticRegression(max_iter=2000,
                                   class_weight="balanced")).fit(Xl, yl)
for nama, m in [("biasa", biasa), ("balanced", seimbang)]:
    p = m.predict_proba(Xu)[:, 1]
    print(f"{nama:8s}: rata-rata peluang {p.mean():.3f} "
          f"(porsi churn {yu.mean():.3f})")

p = biasa.predict_proba(Xu)[:, 1]
ambang = np.linspace(0.05, 0.9, 86)
f1 = [f1_score(yu, p >= t) for t in ambang]
laba = [((p >= t) & (yu == 1)).sum() * 39_000
        - ((p >= t) & (yu == 0)).sum() * 15_000 for t in ambang]
for nama, t in [("bawaan", 0.5), ("F1 maksimum", ambang[np.argmax(f1)]),
                ("laba maksimum", ambang[np.argmax(laba)]),
                ("teoretis c_FP/(c_FP+c_FN)", 15 / 54)]:
    tanda = p >= t
    lb = tanda[yu == 1].sum() * 39_000 - tanda[yu == 0].sum() * 15_000
    print(f"{nama:26s} t={t:.3f}: ditandai {tanda.sum():4d}, "
          f"laba Rp{lb / 1e6:5.2f} jt")

## Listing 11.4: Regresi softmax untuk asal biji

`kode/bab11/multikelas.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
rasa = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance",
        "overall"]
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(f"akurasi CV softmax: "
      f"{cross_val_score(m, biji[rasa], biji['asal'], cv=cv).mean():.3f}")
m.fit(biji[rasa], biji["asal"])
koef = pd.DataFrame(m[-1].coef_, index=m[-1].classes_, columns=rasa)
print(koef.round(2).to_string())
contoh = biji[rasa].iloc[[0]]
p = m.predict_proba(contoh)[0]
print("kiriman pertama:", ", ".join(f"{k} {v:.3f}"
                                    for k, v in zip(m.classes_, p)),
      "| asal sebenarnya:", biji["asal"].iloc[0])

## Listing 11.5: Studi kasus skor churn dan anggaran voucer

`kode/bab11/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder,
                                   StandardScaler)

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan.csv")
df["tren"] = df["frekuensi_90h"] / (df["frekuensi_90h_lalu"] + 1)
miring = ["recency_hari", "frekuensi_90h", "nilai_90h", "jarak_km"]
lain = ["frekuensi_90h_lalu", "rata_nilai_struk", "porsi_promo",
        "porsi_aplikasi", "porsi_kopi_hitam", "porsi_pagi",
        "porsi_akhir_pekan", "ragam_menu", "keluhan_90h",
        "pakai_ulasan", "umur", "tren"]
kat = ["tingkat", "kota", "jenis_kelamin"]


def praproses(log):
    m = (make_pipeline(FunctionTransformer(np.log1p,
                                           feature_names_out="one-to-one"),
                       StandardScaler()) if log else StandardScaler())
    return ColumnTransformer([("miring", m, miring),
                              ("num", StandardScaler(), lain),
                              ("kat", OneHotEncoder(), kat)])


Xl, Xu, yl, yu = train_test_split(df, df["churn"], test_size=0.3,
                                  stratify=df["churn"], random_state=7)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for log in [False, True]:
    s = cross_val_score(make_pipeline(praproses(log), LogisticRegression(
        max_iter=3000)), Xl, yl, cv=cv, scoring="roc_auc").mean()
    print(f"log1p pada atribut menceng = {log!s:5s}: AUC CV {s:.4f}")

model = make_pipeline(praproses(False), LogisticRegressionCV(
    Cs=np.logspace(-3, 2, 11), cv=cv, scoring="neg_log_loss",
    max_iter=3000))
model.fit(Xl, yl)
lr = model[-1]
p = model.predict_proba(Xu)[:, 1]
print(f"C terpilih {lr.C_[0]:.3f}; uji: AUC {roc_auc_score(yu, p):.3f}, "
      f"Brier {brier_score_loss(yu, p):.4f}")

nama = [n.split("__")[1] for n in model[0].get_feature_names_out()]
koef = pd.Series(lr.coef_[0], index=nama)
print("\nkoefisien terbesar (per simpangan baku):")
print(koef[koef.abs().sort_values(ascending=False).index[:8]].round(2)
      .to_string())

skor = model.predict_proba(df)[:, 1]
sasaran = skor >= 15 / 54
print(f"\nskor >= 0.278: {sasaran.mean():.1%} anggota "
      f"(~{sasaran.mean() * 26000:.0f} dari 26.000)")
laba = 0.3 * 180_000 * skor[sasaran].sum() - 15_000 * sasaran.sum()
print(f"harapan laba: Rp{laba / 1e6:.1f} jt per 8.000 anggota, "
      f"Rp{laba / 1e6 * 26000 / 8000:.1f} jt per 26.000")
print("sasaran per tingkat:", df[sasaran].groupby("tingkat").size()
      .to_dict())

Z = model[0].transform(df)
i = int(np.argsort(-skor)[500])
sumbang = pd.Series(Z[i] * lr.coef_[0], index=nama)
print(f"\nanggota {df['id_anggota'].iloc[i]}, skor {skor[i]:.2f}; "
      "tiga alasan teratas:")
print(sumbang.sort_values(ascending=False).head(3).round(2).to_string())
print(df.loc[i, ["recency_hari", "tren", "keluhan_90h", "tingkat"]]
      .to_dict())